# ALS dengan bias: pembaruan tertutup bergantian

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imamrozi/recsys-hibrida-multisinyal-notebooks/blob/main/notebooks/bab04_hybrid_cf/01_als_dengan_bias.ipynb)

**Buku pendamping:** *Sistem Rekomendasi Hibrida Multi-Sinyal: Rating, Sentimen, dan Emosi* · Bab 4 · Kode 4.2.1

**Tujuan belajar**

- Memahami bahwa ALS menyelesaikan tiap faktor sebagai regresi ridge (solusi tertutup).
- Menguraikan sumbangan rata-rata global, bias, dan faktor laten terhadap akurasi.
- Menyetel dimensi d dan regularisasi λ pada data validasi.

> **Catatan.** Semua data di notebook ini **simulasi** dan dibuat agar mudah dijalankan (CPU, hitungan detik). Angka yang keluar menunjukkan *perilaku metode*, bukan hasil empiris pada dataset nyata, dan tidak boleh dikutip sebagai bukti. Kode bersifat ilustratif dan **belum production-ready** (tanpa validasi masukan, tanpa uji beban).

> Implementasi ALS di buku ditulis dengan perulangan Python agar mudah dibaca. Itu **lambat dan tidak production-ready** untuk data besar. Untuk skala nyata, gunakan pustaka ALS terdistribusi atau vektorisasi.

## 1. Kode dari buku

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def als_bias(R, n_users, n_items, d=50, lam=0.02, iters=10, seed=0):
    """R: daftar (u, i, r) dengan indeks bilangan bulat dari data latih."""
    rng = np.random.default_rng(seed)
    mu = np.mean([r for _, _, r in R])
    bu, bi = np.zeros(n_users), np.zeros(n_items)
    P = rng.normal(0, 0.1, (n_users, d))
    Q = rng.normal(0, 0.1, (n_items, d))

    by_user, by_item = {}, {}
    for u, i, r in R:
        by_user.setdefault(u, []).append((i, r))
        by_item.setdefault(i, []).append((u, r))

    for _ in range(iters):
        # Persamaan (4-3): bias pengguna
        for u, items in by_user.items():
            bu[u] = sum(r - mu - bi[i] - P[u] @ Q[i] for i, r in items) / (len(items) + lam)
        # Bias item, dengan peran pengguna dan item dipertukarkan
        for i, users in by_item.items():
            bi[i] = sum(r - mu - bu[u] - P[u] @ Q[i] for u, r in users) / (len(users) + lam)
        # Persamaan (4-4): faktor pengguna sebagai solusi ridge
        for u, items in by_user.items():
            Qu = Q[[i for i, _ in items]]
            y = np.array([r - mu - bu[u] - bi[i] for i, r in items])
            P[u] = np.linalg.solve(Qu.T @ Qu + lam * np.eye(d), Qu.T @ y)
        # Faktor item, simetris dengan faktor pengguna
        for i, users in by_item.items():
            Pi = P[[u for u, _ in users]]
            y = np.array([r - mu - bu[u] - bi[i] for u, r in users])
            Q[i] = np.linalg.solve(Pi.T @ Pi + lam * np.eye(d), Pi.T @ y)
    return mu, bu, bi, P, Q

## 2. Data simulasi

500 pengguna × 200 item dengan struktur laten berpangkat 4, bias pengguna dan item, derau, dan kepadatan 20%. Kita sembunyikan 20% rating untuk uji.

In [2]:
def simulasi_laten(m=500, n=200, rank=4, density=0.20, noise=0.5, seed=11):
    rng = np.random.default_rng(seed)
    P = rng.normal(0, 1, (m, rank)); Q = rng.normal(0, 1, (n, rank))
    bu = rng.normal(0, 0.5, m)[:, None]; bi = rng.normal(0, 0.5, n)[None, :]
    nilai = 3.4 + bu + bi + 0.6 * P @ Q.T + rng.normal(0, noise, (m, n))
    nilai = np.clip(np.round(nilai), 1, 5)
    u, i = np.nonzero(rng.random((m, n)) < density)
    return u, i, nilai[u, i], m, n

u, i, r, m, n = simulasi_laten()
perm = np.random.default_rng(0).permutation(len(r))
cut = int(0.2 * len(r))
te, tr = perm[:cut], perm[cut:]
R_latih = list(zip(u[tr], i[tr], r[tr]))
print("rating latih:", len(tr), "| uji:", len(te), "| kepadatan: %.1f%%" % (100 * len(r) / (m * n)))

def pred(model, uu, ii, faktor=True):
    mu, bu, bi, P, Q = model
    out = mu + bu[uu] + bi[ii]
    return out + (P[uu] * Q[ii]).sum(1) if faktor else out

def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

rating latih: 15795 | uji: 3948 | kepadatan: 19.7%


## 3. Eksperimen A: sumbangan tiap komponen

Latih ALS (*d* = 8, λ = 5) lalu bandingkan RMSE uji: hanya rata-rata global, rata-rata global + bias, dan model penuh. λ di sini lebih besar daripada nilai awal buku karena data kecil dan berderau.

In [3]:
model = als_bias(R_latih, m, n, d=8, lam=5.0, iters=10)
mu = model[0]
rows = {
    "rata-rata global": rmse(r[te], np.full(len(te), mu)),
    "+ bias pengguna dan item": rmse(r[te], pred(model, u[te], i[te], faktor=False)),
    "+ faktor laten (model penuh)": rmse(r[te], pred(model, u[te], i[te])),
}
print(pd.Series(rows, name="RMSE uji").round(3))
assert rows["+ bias pengguna dan item"] < rows["rata-rata global"]
assert rows["+ faktor laten (model penuh)"] < rows["+ bias pengguna dan item"]

rata-rata global                1.245
+ bias pengguna dan item        1.110
+ faktor laten (model penuh)    0.693
Name: RMSE uji, dtype: float64


**Interpretasi (simulasi).** Pada simulasi ini faktor laten menyumbang penurunan galat terbesar, karena struktur laten sengaja dibuat kuat (bobot 0,6 pada perkalian faktor). Di data nyata biasanya sebaliknya: baseline bias sering sudah mengambil porsi besar, dan faktor laten memberi tambahan yang lebih kecil. Karena itu baseline bias tetap wajib dilaporkan, agar kontribusi faktor laten terukur dengan jujur.

## 4. Eksperimen B: jumlah iterasi

Latih ulang dengan `iters` berbeda dan lihat galat latih dan uji.

In [4]:
rows = []
for it in (1, 2, 3, 5, 10):
    mdl = als_bias(R_latih, m, n, d=8, lam=5.0, iters=it)
    rows.append({"iters": it, "RMSE_latih": rmse(r[tr], pred(mdl, u[tr], i[tr])),
                 "RMSE_uji": rmse(r[te], pred(mdl, u[te], i[te]))})
hasil_it = pd.DataFrame(rows).set_index("iters")
print(hasil_it.round(3))
assert hasil_it["RMSE_latih"].iloc[-1] < hasil_it["RMSE_latih"].iloc[0]

       RMSE_latih  RMSE_uji
iters                      
1           0.989     1.099
2           0.636     0.847
3           0.529     0.715
5           0.514     0.691
10          0.506     0.693


**Interpretasi.** ALS menurunkan galat latih dengan cepat: setiap langkah adalah solusi tertutup per pengguna atau item, bukan langkah kecil gradien. Pada data ini galat berhenti turun setelah sekitar lima iterasi. Perhatikan galat uji, bukan hanya galat latih, saat memutuskan berhenti.

## 5. Eksperimen C: dimensi d dan regularisasi λ

Grid kecil dengan `iters=6` agar cepat. Pilih kombinasi terbaik pada **validasi**, lalu laporkan pada uji. Di sini validasi diambil dari sebagian data latih.

In [5]:
perm_tr = np.random.default_rng(1).permutation(len(tr))
va_idx, fit_idx = tr[perm_tr[:len(tr) // 5]], tr[perm_tr[len(tr) // 5:]]
R_fit = list(zip(u[fit_idx], i[fit_idx], r[fit_idx]))

rows = []
for d in (2, 4, 8, 16):
    for lam in (0.5, 2.0, 5.0, 15.0):
        mdl = als_bias(R_fit, m, n, d=d, lam=lam, iters=6)
        rows.append({"d": d, "lam": lam, "RMSE_validasi": rmse(r[va_idx], pred(mdl, u[va_idx], i[va_idx]))})
grid = pd.DataFrame(rows).pivot(index="d", columns="lam", values="RMSE_validasi")
print(grid.round(3))
d_best, lam_best = [(d, l) for d in grid.index for l in grid.columns if grid.loc[d, l] == grid.values.min()][0]
final = als_bias(R_latih, m, n, d=d_best, lam=lam_best, iters=6)
print(f"terbaik pada validasi: d={d_best}, lam={lam_best} | RMSE uji = {rmse(r[te], pred(final, u[te], i[te])):.3f}")

lam   0.5    2.0    5.0    15.0
d                              
2    0.948  0.936  0.936  1.056
4    0.665  0.658  0.701  1.030
8    0.811  0.716  0.717  1.004
16   1.038  0.777  0.741  0.983
terbaik pada validasi: d=4, lam=2.0 | RMSE uji = 0.645


**Interpretasi.** λ yang terlalu kecil membuat solusi ridge menghafal (terutama pada pengguna dengan sedikit rating), dan galat validasi naik pada *d* besar. λ yang terlalu besar membuat model *underfit*. Nilai menengah menstabilkan faktor. Dimensi *d* lebih besar hanya membantu bila λ ikut memadai. Menyetel *d* dan λ bersama pada validasi, lalu melaporkan satu kali pada uji, adalah protokol yang benar.

## Pertanyaan refleksi

1. Mengapa solusi tertutup `np.linalg.solve(Qu.T @ Qu + lam * I, Qu.T @ y)` stabil untuk pengguna dengan satu rating, padahal `Qu` hanya satu baris?
2. Dengan `lam=0`, apa yang terjadi pada pengguna yang jumlah ratingnya lebih sedikit daripada d?
3. Bagaimana ALS diparalelkan, dan mengapa langkah per pengguna (dan per item) saling bebas?